# Arnowitt-Deser-Misner (ADM) and Baumgarte-Shapiro-Shibata-Nakamura (BSSN) Conversions

Follow the ADM-to-BSSN conversion pipeline, then check that a toy data set can be
converted forward and reconstructed back to ADM fields.

Navigation:
[Index](../index.ipynb) |
Previous: [ADM 3+1 Background](adm_3plus1_background.ipynb) |
Next: [Index](../index.ipynb)

## Learning Goals

- Convert ADM variables into BSSN variables used for stable numerical evolution.
- Understand the conformal factor as a separated scale of the spatial metric.
- Check that converting to BSSN and back preserves a toy data set.

## Words for This Notebook

| Name | Plain Meaning | Code Name |
| --- | --- | --- |
| ADM | spacetime split into space plus time | `gammaDD`, `KDD`, `betaU` |
| BSSN | evolution-variable rewrite of ADM fields | `cf`, `hDD`, `aDD` |
| ADM metric | spatial distances on one time slice | `gammaDD` |
| ADM curvature | how the slice bends through spacetime | `KDD` |
| ADM shift | coordinate sliding within the slice | `betaU` |
| reference metric | coordinate-system baseline geometry | `ghatDD` |
| reference determinant | determinant of the reference metric | `detgammahat` |
| reference rescaling factors | scale factors removed from components | `ReDD`, `ReU` |
| conformal metric | spatial metric after scale removal | `gammabarDD` |
| conformal factor | the removed scale variable | `cf` |
| W | conformal-factor choice equal to `exp(-2 phi)` | `EvolvedConformalFactor_cf` |
| metric correction | conformal metric minus the reference metric | `hDD` |
| trace-free curvature | curvature with its trace removed | `aDD` |
| curvature trace | contraction of `KDD` with the inverse metric | `trK` |
| rescaled shift | shift divided by the reference scale factor | `vetU` |
| residual | expression that should simplify to zero | `sp.factor(...)` |

Each output below is framed as evidence for one part of the conversion pipeline.

## Step 1: Map the ADM-to-BSSN Conversion

The BSSN formulation rewrites ADM fields into variables better suited for
numerical evolution. This notebook uses Brown's covariant BSSN idea
([2009](https://arxiv.org/abs/0902.3652)) and the reference-metric conventions
used by Baumgarte and collaborators ([2012](https://arxiv.org/abs/1211.6632))
and by Ruchlin, Etienne, and Baumgarte
([2018](https://arxiv.org/abs/1712.07658)).

Read the conversion as a pipeline:

| Stage | Input | Output to inspect |
| --- | --- | --- |
| ADM data | `gammaDD`, `KDD`, `betaU`, `BU` | physical metric, curvature, shift |
| Scale split | determinant ratio `detgammahat / detgamma` | `cf`, `gammabarDD` |
| Reference normalization | `ReDD`, `ReU` factors | `hDD`, `aDD`, `vetU`, `betU` |
| Curvature split | `KDD` and inverse metric | `trK`, trace-free `aDD` |
| Inverse check | computed BSSN symbols | reconstructed `gammaDD`, `KDD`, `betaU` |

For the conformal metric, NRPy uses the reference metric
`ghatDD` and its determinant `detgammahat`:

$$
\bar{\gamma}_{ij}
= \left(\frac{\hat{\gamma}}{\gamma}\right)^{1/3}\gamma_{ij},
\qquad
h_{ij}
= \frac{\bar{\gamma}_{ij}-\hat{\gamma}_{ij}}{\mathrm{ReDD}_{ij}}.
$$

For the curvature variables, it separates the trace from the trace-free part:

$$
K=\gamma^{ij}K_{ij},
\qquad
\bar{A}_{ij}
= \left(\frac{\hat{\gamma}}{\gamma}\right)^{1/3}
  \left(K_{ij}-\frac{1}{3}\gamma_{ij}K\right),
\qquad
a_{ij}=\frac{\bar{A}_{ij}}{\mathrm{ReDD}_{ij}}.
$$

With the `W` conformal-factor choice,

$$
W=\left(\frac{\gamma}{\hat{\gamma}}\right)^{-1/6},
\qquad
\mathrm{vet}^i=\frac{\beta^i}{\mathrm{ReU}_i}.
$$

## Setup: Import Conversion Tools

The next cell imports symbolic arrays, runtime parameters, and the two conversion
classes used in the round-trip check.

In [1]:
import sympy as sp

import nrpy.indexedexp as ixp
import nrpy.params as par
from nrpy.equations.general_relativity.ADM_to_BSSN import ADM_to_BSSN
from nrpy.equations.general_relativity.BSSN_to_ADM import BSSN_to_ADM
from nrpy.equations.general_relativity.InitialData_Cartesian import (
    InitialData_Cartesian,
)

## Step 2: Choose the Conformal-Factor Variable

NRPy can evolve several equivalent forms of the same metric scale. This notebook
uses `W`, the current default, because conformally flat data have the simple
expectation `cf = psi**(-2)`.

| Choice | Meaning | Conformally flat expectation |
| --- | --- | --- |
| `phi` | logarithmic scale | `log(psi)` |
| `chi` | `exp(-4 phi)` | `psi**(-4)` |
| `W` | `exp(-2 phi)` | `psi**(-2)` |

The next cell fixes that runtime setting and prints it before any conversion is
run.

In [2]:
par.set_parval_from_str("EvolvedConformalFactor_cf", "W")
print("conformal factor choice:", par.parval_from_str("EvolvedConformalFactor_cf"))

conformal factor choice: W


## Step 3: Convert Brill-Lindquist ADM Data

The Brill-Lindquist
[initial-data construction](https://doi.org/10.1103/PhysRev.131.471) is
conformally flat with zero curvature and zero shift. For this example, inspect
whether the scale information remains in `cf` while `hDD`, `aDD`, `trK`, `vetU`,
and `betU` all reduce to zero residuals.

In [3]:
brill_lindquist = InitialData_Cartesian("BrillLindquist")
adm_to_bssn = ADM_to_BSSN(
    brill_lindquist.gammaDD,
    brill_lindquist.KDD,
    brill_lindquist.betaU,
    brill_lindquist.BU,
    "Cartesian",
)
brill_hDD_residuals = [
    sp.factor(adm_to_bssn.hDD[i][j])
    for i in range(3)
    for j in range(3)
]
brill_aDD_residuals = [
    sp.factor(adm_to_bssn.aDD[i][j])
    for i in range(3)
    for j in range(3)
]
brill_shift_residuals = adm_to_bssn.vetU + adm_to_bssn.betU
brill_cf_symbols = sorted(str(symbol) for symbol in adm_to_bssn.cf.free_symbols)

print("cf depends on:", brill_cf_symbols)
print("hDD residuals:", brill_hDD_residuals)
print("aDD residuals:", brill_aDD_residuals)
print("trK residual:", sp.factor(adm_to_bssn.trK))
print("shift residuals:", brill_shift_residuals)
if (
    brill_hDD_residuals != [0] * 9
    or brill_aDD_residuals != [0] * 9
    or sp.factor(adm_to_bssn.trK) != 0
    or brill_shift_residuals != [0] * 6
):
    raise RuntimeError("Expected Brill-Lindquist BSSN residuals to vanish.")

Setting up reference_metric[Cartesian]...
cf depends on: ['BH1_mass', 'BH1_posn_x', 'BH1_posn_y', 'BH1_posn_z', 'BH2_mass', 'BH2_posn_x', 'BH2_posn_y', 'BH2_posn_z', 'x', 'y', 'z']
hDD residuals: [0, 0, 0, 0, 0, 0, 0, 0, 0]
aDD residuals: [0, 0, 0, 0, 0, 0, 0, 0, 0]
trK residual: 0
shift residuals: [0, 0, 0, 0, 0, 0]


## Step 4: Build a Toy ADM Test Case

The trusted test data are a conformally flat metric, symbolic curvature, symbolic
shift, and zero shift derivative:

$$
\gamma_{ij}=\psi^4\delta_{ij},
\qquad
K_{xx}=k_{xx},\ K_{xy}=K_{yx}=k_{xy},\ K_{yy}=k_{yy},\ K_{zz}=k_{zz},
\qquad
\beta^i=(\beta_0,\beta_1,\beta_2).
$$

This test keeps the metric simple enough to inspect while making curvature and
shift nonzero. That lets the round-trip check exercise more than the
Brill-Lindquist zero-curvature case.

In [4]:
psi = sp.symbols("psi", positive=True)
kxx, kxy, kyy, kzz = sp.symbols("kxx kxy kyy kzz")
beta0, beta1, beta2 = sp.symbols("beta0 beta1 beta2")

toy_gammaDD = ixp.zerorank2()
toy_KDD = ixp.zerorank2()
toy_betaU = ixp.zerorank1()
toy_BU = ixp.zerorank1()
for i in range(3):
    toy_gammaDD[i][i] = psi**4
toy_KDD[0][0] = kxx
toy_KDD[0][1] = toy_KDD[1][0] = kxy
toy_KDD[1][1] = kyy
toy_KDD[2][2] = kzz
toy_betaU[0] = beta0
toy_betaU[1] = beta1
toy_betaU[2] = beta2

print("toy gammaDD diagonal:", [toy_gammaDD[i][i] for i in range(3)])
print("toy KDD nonzero entries:", [kxx, kxy, kyy, kzz])
print("toy betaU:", toy_betaU)
print("toy BU:", toy_BU)

toy gammaDD diagonal: [psi**4, psi**4, psi**4]
toy KDD nonzero entries: [kxx, kxy, kyy, kzz]
toy betaU: [beta0, beta1, beta2]
toy BU: [0, 0, 0]


## Step 5: Run the Forward Conversion on Toy ADM Data

The trusted input is the toy ADM data written above. The new result is computed
by `ADM_to_BSSN(...)`. The residuals check the expected conformal factor,
curvature trace, conformal-metric correction, and rescaled shift. The printed
`aDD[0][0]` entry is not expected to vanish; it carries trace-free curvature.

In [5]:
toy_adm_to_bssn = ADM_to_BSSN(
    toy_gammaDD,
    toy_KDD,
    toy_betaU,
    toy_BU,
    "Cartesian",
)

expected_trK = (kxx + kyy + kzz) / psi**4
cf_residual = sp.factor(toy_adm_to_bssn.cf - psi**(-2))
trK_residual = sp.factor(toy_adm_to_bssn.trK - expected_trK)
hDD_residuals = [
    sp.factor(toy_adm_to_bssn.hDD[i][j])
    for i in range(3)
    for j in range(3)
]
vetU_residuals = [
    sp.factor(toy_adm_to_bssn.vetU[i] - toy_betaU[i])
    for i in range(3)
]

print("forward cf residual:", cf_residual)
print("forward trK residual:", trK_residual)
print("forward hDD residuals:", hDD_residuals)
print("forward vetU residuals:", vetU_residuals)
print("toy aDD[0][0]:", toy_adm_to_bssn.aDD[0][0])
if (
    cf_residual != 0
    or trK_residual != 0
    or hDD_residuals != [0] * 9
    or vetU_residuals != [0, 0, 0]
):
    raise RuntimeError("Expected the forward ADM-to-BSSN residuals to vanish.")

forward cf residual: 0
forward trK residual: 0
forward hDD residuals: [0, 0, 0, 0, 0, 0, 0, 0, 0]
forward vetU residuals: [0, 0, 0]
toy aDD[0][0]: (kxx - psi**4*(kxx/psi**4 + kyy/psi**4 + kzz/psi**4)/3)/psi**4


## Step 6: Build the Inverse Substitution Map

`BSSN_to_ADM("Cartesian")` builds symbolic inverse formulas containing BSSN
symbols. The substitution map below is the bridge in the round-trip test: every
inverse-formula symbol is replaced by the value computed by `ADM_to_BSSN`, not
by a hand-written expected answer.

In [6]:
toy_bssn_to_adm = BSSN_to_ADM("Cartesian")
substitution_sources = []
for i in range(3):
    substitution_sources.extend(toy_bssn_to_adm.gammaDD[i])
    substitution_sources.extend(toy_bssn_to_adm.KDD[i])
substitution_sources.extend(toy_bssn_to_adm.betaU)


def forward_value_for(symbol_name):
    if symbol_name == "cf":
        return toy_adm_to_bssn.cf
    if symbol_name == "trK":
        return toy_adm_to_bssn.trK
    if symbol_name.startswith("vetU"):
        return toy_adm_to_bssn.vetU[int(symbol_name[-1])]
    if symbol_name.startswith("hDD"):
        return toy_adm_to_bssn.hDD[int(symbol_name[-2])][int(symbol_name[-1])]
    if symbol_name.startswith("aDD"):
        return toy_adm_to_bssn.aDD[int(symbol_name[-2])][int(symbol_name[-1])]
    raise ValueError(f"Unexpected BSSN symbol: {symbol_name}")


substitutions = {
    symbol: forward_value_for(symbol.name)
    for expression in substitution_sources
    for symbol in expression.free_symbols
}
print("substituted BSSN symbols:", sorted(symbol.name for symbol in substitutions))

Setting up BSSN_Quantities[Cartesian]...


substituted BSSN symbols: ['aDD00', 'aDD01', 'aDD02', 'aDD11', 'aDD12', 'aDD22', 'cf', 'hDD00', 'hDD01', 'hDD02', 'hDD11', 'hDD12', 'hDD22', 'trK', 'vetU0', 'vetU1', 'vetU2']


## Validation Check: Compare Reconstructed ADM Fields

Finally, substitute the computed BSSN values into the inverse formulas and compare
every reconstructed ADM component against the trusted toy data. A zero residual
means the forward values carry enough information to recover the original ADM
metric, curvature, shift, and metric determinant.

In [7]:
metric_residuals = [
    sp.factor(toy_bssn_to_adm.gammaDD[i][j].xreplace(substitutions) - toy_gammaDD[i][j])
    for i in range(3)
    for j in range(3)
]
curvature_residuals = [
    sp.factor(toy_bssn_to_adm.KDD[i][j].xreplace(substitutions) - toy_KDD[i][j])
    for i in range(3)
    for j in range(3)
]
shift_residuals = [
    sp.factor(toy_bssn_to_adm.betaU[i].xreplace(substitutions) - toy_betaU[i])
    for i in range(3)
]
reconstructed_gammaDD = [
    [
        sp.factor(toy_bssn_to_adm.gammaDD[i][j].xreplace(substitutions))
        for j in range(3)
    ]
    for i in range(3)
]
_toy_gammaUU, toy_gamma_det = ixp.symm_matrix_inverter3x3(toy_gammaDD)
_reconstructed_gammaUU, reconstructed_gamma_det = ixp.symm_matrix_inverter3x3(
    reconstructed_gammaDD
)
determinant_residual = sp.factor(reconstructed_gamma_det - toy_gamma_det)

print("metric residuals:", metric_residuals)
print("KDD residuals:", curvature_residuals)
print("shift residuals:", shift_residuals)
print("determinant residual:", determinant_residual)
if (
    metric_residuals != [0] * 9
    or curvature_residuals != [0] * 9
    or shift_residuals != [0, 0, 0]
    or determinant_residual != 0
):
    raise RuntimeError("Expected all ADM-BSSN round-trip residuals to vanish.")

metric residuals: [0, 0, 0, 0, 0, 0, 0, 0, 0]
KDD residuals: [0, 0, 0, 0, 0, 0, 0, 0, 0]
shift residuals: [0, 0, 0]
determinant residual: 0


The forward residuals check the BSSN variables computed from the toy ADM data.
The substitution map ensures the inverse formulas use those computed values. The
reconstruction and determinant residuals then check that the original ADM metric,
curvature, and shift are recovered.

## Learning Check

After running the round-trip check, explain why using the computed BSSN values in
the substitution map is stronger than substituting expected zeros by hand.

## Continue

Next: [Index](../index.ipynb)